In [1]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/crop_disease_dataset"


# loading the data
from datasets import load_from_disk

train_dataset = load_from_disk(f"{SAVE_PATH}/train")
valid_dataset = load_from_disk(f"{SAVE_PATH}/valid")


In [ ]:
print("Train:", len(train_dataset))
print("Valid:", len(valid_dataset))

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
import timm

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [ ]:
class PlantDataset(torch.utils.data.Dataset):

    def __init__(self, hf_dataset, transform=None):

        self.ds = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):

        image = self.ds[idx]["image"].convert("RGB")
        label = self.ds[idx]["label"]

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
train_ds = PlantDataset(
    train_dataset,
    train_transform
)

val_ds = PlantDataset(
    valid_dataset,
    val_transform
)

train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=32,
    shuffle=False,
)

In [ ]:
img, label = train_ds[0]

print("dtype:", img.dtype)
print("min:", img.min().item())
print("max:", img.max().item())

In [ ]:
num_classes = len(
    set(train_dataset['label_name'])
)

print(num_classes)

In [ ]:
import torch
import torch.nn as nn

class Encoder(nn.Module):

    def __init__(self, latent_dim):

        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(3, 32, 3, stride=2, padding=1),
            nn.ReLU(),

            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(),

            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.ReLU(),

            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.ReLU()
        )

        self.flatten = nn.Flatten()

        self.fc = nn.Linear(
            256 * 14 * 14,
            latent_dim
        )

    def forward(self, x):

        x = self.conv(x)

        x = self.flatten(x)

        z = self.fc(x)

        return z

In [ ]:
class Decoder(nn.Module):

    def __init__(self, latent_dim):

        super().__init__()

        self.fc = nn.Linear(
            latent_dim,
            256 * 14 * 14
        )

        self.decoder = nn.Sequential(

            nn.ConvTranspose2d(
                256, 128,
                kernel_size=4,
                stride=2,
                padding=1
            ),
            nn.ReLU(),

            nn.ConvTranspose2d(
                128, 64,
                kernel_size=4,
                stride=2,
                padding=1
            ),
            nn.ReLU(),

            nn.ConvTranspose2d(
                64, 32,
                kernel_size=4,
                stride=2,
                padding=1
            ),
            nn.ReLU(),

            nn.ConvTranspose2d(
                32, 3,
                kernel_size=4,
                stride=2,
                padding=1
            ),

            nn.Sigmoid()

        )

    def forward(self, z):

        x = self.fc(z)

        x = x.view(
            -1,
            256,
            14,
            14
        )

        x = self.decoder(x)

        return x

In [ ]:
class AutoEncoder(nn.Module):

    def __init__(self, latent_dim):

        super().__init__()

        self.encoder = Encoder(latent_dim)

        self.decoder = Decoder(latent_dim)

    def forward(self, x):

        z = self.encoder(x)

        reconstruction = self.decoder(z)

        return reconstruction

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

latent_dim = 32

model = AutoEncoder(
    latent_dim=latent_dim
).to(device)


print(model)

In [ ]:
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print(f"Trainable parameters: {trainable_params:,}")

In [ ]:

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

In [ ]:
import torch.nn.functional as F

def reconstruction_loss(reconstruction, original):

    loss = F.mse_loss(
        reconstruction,
        original,
        reduction="mean"
    )

    return loss

In [ ]:
from tqdm.auto import tqdm

def validate_one_epoch(
        model,
        dataloader):

    model.eval()

    running_loss = 0

    with torch.no_grad():

        pbar = tqdm(
            dataloader,
            desc="Validation",
            leave=False
        )

        for images, labels in pbar:

            images = images.to(device)

            reconstruction = model(images)

            loss = reconstruction_loss(
                reconstruction,
                images
            )

            running_loss += loss.item()

            pbar.set_postfix({
                "Val Loss": f"{loss.item():.6f}"
            })

    epoch_loss = running_loss / len(dataloader)

    return epoch_loss

In [ ]:
def train_one_epoch(
        model,
        dataloader,
        optimizer):

    model.train()

    running_loss = 0

    pbar = tqdm(
        dataloader,
        desc="Training",
        leave=False
    )

    for images, labels in pbar:

        images = images.to(device)

        optimizer.zero_grad()

        reconstruction = model(images)

        loss = reconstruction_loss(
            reconstruction,
            images
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        pbar.set_postfix({
            "Loss": f"{loss.item():.6f}"
        })

    epoch_loss = running_loss / len(dataloader)

    return epoch_loss

In [ ]:
save_dir = "/content/drive/MyDrive/CropDiseaseModels"

In [ ]:
epochs = 100
patience = 5

train_loss_history = []
val_loss_history = []

best_val_loss = float("inf")
best_epoch = 0

counter = 0

for epoch in range(epochs):

    print(f"\nEpoch {epoch+1}/{epochs}")

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer
    )

    val_loss = validate_one_epoch(
        model,
        val_loader
    )

    train_loss_history.append(train_loss)
    val_loss_history.append(val_loss)

    print(
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f}"
    )


    if val_loss < best_val_loss:
      best_val_loss = val_loss
      best_epoch = epoch + 1

      counter = 0

      torch.save(
            model.state_dict(),
            f"{save_dir}/autoencoder_model_weights_v3.pth"
                  )

      torch.save(
          model,
          f"{save_dir}/autoencoder_full_model_v3.pth"
      )
      print("Model saved successfully.")

    else:

        counter += 1

        print(
            f"No improvement for {counter} epoch(s)."
        )

    # Early stopping

    if counter >= patience:

        print("\nEarly stopping triggered.")
        break

print(f"\nBest validation loss : {best_val_loss:.6f}")
print(f"Best epoch           : {best_epoch}")